# 🚗 Indian Used Car Price Prediction (Regression)

**College ML Mini-Project**
- **Dataset**: Vehicle Dataset from CarDekho ()
- **Source**: [Kaggle](https://www.kaggle.com/datasets/nehalbirla/vehicle-dataset-from-cardekho)
- **License**: Open Data Commons PDDL / CC0
- **Currency**: Indian Rupees (INR / ₹)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")


## 1. Load Real Dataset

In [ ]:
data_path = Path("../data/car_details_v3.csv")
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

## 2. Data Cleaning & Feature Extraction

In [ ]:
# Extract numeric specs from strings
df["engine_cc"] = pd.to_numeric(df["engine"].astype(str).str.extract(r"([0-9.]+)")[0], errors="coerce")
df["max_power_bhp"] = pd.to_numeric(df["max_power"].astype(str).str.extract(r"([0-9.]+)")[0], errors="coerce")
df["mileage_kmpl"] = pd.to_numeric(df["mileage"].astype(str).str.extract(r"([0-9.]+)")[0], errors="coerce")
df["brand"] = df["name"].apply(lambda x: str(x).split()[0].title())
df["car_age"] = 2020 - df["year"]

# Impute missing values with medians
for col in ["engine_cc", "max_power_bhp", "mileage_kmpl", "seats"]:
    df[col] = df[col].fillna(df[col].median())

# Deduplicate and trim extreme anomalies
df = df.drop_duplicates().reset_index(drop=True)
q1 = df["selling_price"].quantile(0.01)
q99 = df["selling_price"].quantile(0.99)
df = df[(df["selling_price"] >= q1) & (df["selling_price"] <= q99)].reset_index(drop=True)
print(f"Cleaned Dataset: {df.shape[0]} rows × {df.shape[1]} columns")


## 3. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df["selling_price"] / 100000, kde=True, ax=axes[0], color="#3498db")
axes[0].set_title("Price Distribution (in Lakhs ₹)")
axes[0].set_xlabel("Price (Lakhs ₹)")

top_brands = df["brand"].value_counts().head(8).index
brand_avg = df[df["brand"].isin(top_brands)].groupby("brand")["selling_price"].mean() / 100000
sns.barplot(x=brand_avg.values, y=brand_avg.index, ax=axes[1], palette="viridis")
axes[1].set_title("Average Price by Top Brands (Lakhs ₹)")
axes[1].set_xlabel("Price (Lakhs ₹)")
plt.tight_layout()
plt.show()


## 4. Model Training & Comparison

In [ ]:
num_cols = ["year", "km_driven", "max_power_bhp"]
cat_cols = ["brand", "fuel", "transmission"]
features = num_cols + cat_cols

X = df[features]
y = df["selling_price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
])

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=10.0),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, random_state=42)
}

results = []
for name, m in models.items():
    pipe = Pipeline([("prep", preprocessor), ("reg", m)])
    cv_mae = -cross_val_score(pipe, X_train, y_train, cv=5, scoring="neg_mean_absolute_error").mean()
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    results.append({
        "Model": name,
        "CV MAE (₹)": round(cv_mae),
        "Test MAE (₹)": round(mean_absolute_error(y_test, preds)),
        "Test R²": round(r2_score(y_test, preds), 4)
    })

res_df = pd.DataFrame(results)
print(res_df.to_string(index=False))
